In [2]:
import sys
print(sys.executable)


/venv/main/bin/python


In [3]:
import sys
!{sys.executable} -m pip install -U pandas


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 88.7 MB/s  0:00:00


In [4]:
import pandas as pd
print(pd.__version__)


3.0.0


In [4]:
import pandas as pd

df = pd.read_csv(
    "df_with_c2s.csv.gz",
    compression="gzip",
    encoding="cp1252",  # یا "latin1"
    low_memory=False
)

print(df.shape)
df.head()


(4398, 9)


,ModelID,drug1_name,drug2_name,label,cell_sentence,cell,drug1_smiles,drug2_smiles,drugpair_smiles
0,ACH-000788,5-FU,ABT-888,synergy,MT-ATP8 MT-CO2 MT-ATP6 MT-CO3 MT-CO1 MT-ND4 MT...,A2058,C1=C(C(=O)NC(=O)N1)F,CC1(CCCN1)C2=NC3=C(C=CC=C3N2)C(=O)N,C1=C(C(=O)NC(=O)N1)F || CC1(CCCN1)C2=NC3=C(C=C...
1,ACH-000788,5-FU,AZD1775,synergy,MT-ATP8 MT-CO2 MT-ATP6 MT-CO3 MT-CO1 MT-ND4 MT...,A2058,C1=C(C(=O)NC(=O)N1)F,CC(C)(C1=NC(=CC=C1)N2C3=NC(=NC=C3C(=O)N2CC=C)N...,C1=C(C(=O)NC(=O)N1)F || CC(C)(C1=NC(=CC=C1)N2C...
2,ACH-000788,5-FU,BEZ-235,synergy,MT-ATP8 MT-CO2 MT-ATP6 MT-CO3 MT-CO1 MT-ND4 MT...,A2058,C1=C(C(=O)NC(=O)N1)F,CC(C)(C#N)C1=CC=C(C=C1)N2C3=C4C=C(C=CC4=NC=C3N...,C1=C(C(=O)NC(=O)N1)F || CC(C)(C#N)C1=CC=C(C=C1...
3,ACH-000788,5-FU,BORTEZOMIB,antagonism,MT-ATP8 MT-CO2 MT-ATP6 MT-CO3 MT-CO1 MT-ND4 MT...,A2058,C1=C(C(=O)NC(=O)N1)F,B(C(CC(C)C)NC(=O)C(CC1=CC=CC=C1)NC(=O)C2=NC=CN...,C1=C(C(=O)NC(=O)N1)F || B(C(CC(C)C)NC(=O)C(CC1...
4,ACH-000788,5-FU,DASATINIB,synergy,MT-ATP8 MT-CO2 MT-ATP6 MT-CO3 MT-CO1 MT-ND4 MT...,A2058,C1=C(C(=O)NC(=O)N1)F,CC1=C(C(=CC=C1)Cl)NC(=O)C2=CN=C(S2)NC3=NC(=NC(...,C1=C(C(=O)NC(=O)N1)F || CC1=C(C(=CC=C1)Cl)NC(=...


In [5]:
from pathlib import Path

# تعریف مسیر ریشه (root)
ROOT = Path.cwd()  # مسیر فعلی دایرکتوری که کد در آن اجرا می‌شود

# بارگذاری داده‌ها
DATA = ROOT / "df_with_c2s.csv.gz"
print("\nDATA exists?", DATA.exists(), "|", DATA)

df = pd.read_csv(DATA, compression="gzip")



DATA exists? True | /workspace/df_with_c2s.csv.gz


In [9]:
# from pathlib import Path
# import pandas as pd

# # ----------------------------
# # 0) ROOT + DATA path (robust)
# # ----------------------------
# try:
#     ROOT  # اگر از قبل تعریف شده باشد
# except NameError:
#     ROOT = Path.cwd()

# # فایل شما طبق خروجی‌تون اینجاست؛ ولی ما robust هم می‌کنیم
# candidates = [
#     ROOT / "df_with_c2s.csv.gz",
#     Path("/home/aibox/df_with_c2s.csv.gz"),
# ]
# DATA = next((p for p in candidates if p.exists()), None)
# if DATA is None:
#     raise FileNotFoundError(
#         "df_with_c2s.csv.gz پیدا نشد. این مسیرها چک شدند:\n" +
#         "\n".join([str(p) for p in candidates])
#     )

# print("\nDATA exists?", DATA.exists(), "|", DATA)

# # ----------------------------
# # 1) Load
# # ----------------------------
# df = pd.read_csv(DATA, compression="gzip")
# print("Loaded shape:", df.shape)
# print("Columns:", list(df.columns))

# # ----------------------------
# # 2) Find correct drug name cols
# # ----------------------------
# def pick_first(existing_cols, candidates_):
#     for c in candidates_:
#         if c in existing_cols:
#             return c
#     return None

# drug1_col = pick_first(df.columns, ["drug1_name", "Drug1", "drug1", "Drug_1", "Drug1_name"])
# drug2_col = pick_first(df.columns, ["drug2_name", "Drug2", "drug2", "Drug_2", "Drug2_name"])

# if drug1_col is None or drug2_col is None:
#     raise KeyError(
#         "ستون‌های اسم دارو پیدا نشد.\n"
#         "ستون‌های موجود:\n" + ", ".join(df.columns) + "\n"
#         "انتظار یکی از این‌ها را داشتیم:\n"
#         "drug1_name/Drug1/... و drug2_name/Drug2/..."
#     )

# # یکدست‌سازی نام ستون‌ها به drug1_name/drug2_name
# df = df.rename(columns={drug1_col: "drug1_name", drug2_col: "drug2_name"})

# # ----------------------------
# # 3) NaN handling + cleanup
# # ----------------------------
# print("\nNaN counts in drug columns:")
# print(df[["drug1_name", "drug2_name"]].isna().sum())

# df[["drug1_name", "drug2_name"]] = (
#     df[["drug1_name", "drug2_name"]]
#     .fillna("")
#     .astype(str)
#     .apply(lambda s: s.str.strip())
# )

# # ----------------------------
# # 4) Sort drug pairs (order-invariant)
# # ----------------------------
# def sort_pair(a: str, b: str):
#     # ترتیب بر اساس casefold برای پایداری (اما خروجی همان رشته اصلی می‌ماند)
#     if a.casefold() <= b.casefold():
#         return a, b
#     return b, a

# df[["drug1_name", "drug2_name"]] = df.apply(
#     lambda r: pd.Series(sort_pair(r["drug1_name"], r["drug2_name"])),
#     axis=1
# )

# # ----------------------------
# # 5) Drop duplicates by drug pair
# # ----------------------------
# df["drug_pair"] = df["drug1_name"].str.cat(df["drug2_name"], sep="_")

# before = len(df)
# df = df.drop_duplicates(subset="drug_pair").drop(columns="drug_pair").reset_index(drop=True)
# after = len(df)

# print(f"\nDropped duplicates: {before - after} rows")
# print("Final shape:", df.shape)

# # ----------------------------
# # 6) Preview
# # ----------------------------
# try:
#     from IPython.display import display
#     display(df.head())
# except Exception:
#     print(df.head())

# # ----------------------------
# # 7) Save gz
# # ----------------------------
# output_path = ROOT / "df_with_c2s_processed.csv.gz"
# df.to_csv(output_path, index=False, compression="gzip")
# print(f"\nFile saved as: {output_path}")



DATA exists? True | /home/aibox/df_with_c2s.csv.gz
Loaded shape: (90800, 23)
Columns: ['ModelID', 'Drug1', 'Drug2', 'classification', 'cell_sentence', 'Cell line', 'Drug1_smilesString', 'Drug2_smilesString', 'drugpair_smiles', 'c2s_cell_type', 'c2s_treated_d1_sentence', 'c2s_treated_d2_sentence', 'c2s_treated_d12_additive', 'c2s_treated_d12_direct', 'c2s_up_d1', 'c2s_down_d1', 'c2s_up_d2', 'c2s_down_d2', 'c2s_up_d12_add', 'c2s_down_d12_add', 'c2s_up_synergy', 'c2s_down_synergy', 'c2s_synergy_score']

NaN counts in drug columns:
drug1_name    0
drug2_name    0
dtype: int64

Dropped duplicates: 86402 rows
Final shape: (4398, 23)


,ModelID,drug1_name,drug2_name,classification,cell_sentence,Cell line,Drug1_smilesString,Drug2_smilesString,drugpair_smiles,c2s_cell_type,...,c2s_treated_d12_direct,c2s_up_d1,c2s_down_d1,c2s_up_d2,c2s_down_d2,c2s_up_d12_add,c2s_down_d12_add,c2s_up_synergy,c2s_down_synergy,c2s_synergy_score
0,ACH-000788,5-FU,ABT-888,synergy,MT-ATP8 MT-CO2 MT-ATP6 MT-CO3 MT-CO1 MT-ND4 MT...,A2058,C1=C(C(=O)NC(=O)N1)F,CC1(CCCN1)C2=NC3=C(C=CC=C3N2)C(=O)N,C1=C(C(=O)NC(=O)N1)F || CC1(CCCN1)C2=NC3=C(C=C...,endothelial cell,...,OP-2500 TP8 CO2 TP6 CO3 CO1 ND4 ND2 ND4L EF1A1...,DR1 ME2 ID1 TK1 CD74 NT5C3B CD320 CD55 SF3A3 E...,GAPDH MT-CYB ACTB SPARC VIM UBB B2M MT2A CD63 ...,DR1 ME2 ID1 TK1 CD74 NT5C3B CD320 CD55 SF3A3 E...,GAPDH MT-CYB ACTB SPARC VIM UBB B2M MT2A CD63 ...,DR1 ME2 ID1 TK1 CD74 NT5C3B CD320 CD55 SF3A3 E...,GAPDH MT-CYB SPARC ACTB VIM UBB B2M MT2A CD63 ...,GAPDH MT-CYB SPARC ACTB VIM UBB FN1 B2M MT2A C...,CD74 NT5C3B CD320 TK1 CD55 SF3A3 EI24 CD96 AP1...,511.354167
1,ACH-000788,5-FU,AZD1775,synergy,MT-ATP8 MT-CO2 MT-ATP6 MT-CO3 MT-CO1 MT-ND4 MT...,A2058,C1=C(C(=O)NC(=O)N1)F,CC(C)(C1=NC(=CC=C1)N2C3=NC(=NC=C3C(=O)N2CC=C)N...,C1=C(C(=O)NC(=O)N1)F || CC(C)(C1=NC(=CC=C1)N2C...,endothelial cell,...,OP-2500 TP8 CO2 TP6 CO3 CO1 ND4 ND2 ND4L EF1A1...,DR1 ME2 ID1 TK1 CD74 NT5C3B CD320 CD55 SF3A3 E...,GAPDH MT-CYB ACTB SPARC VIM UBB B2M MT2A CD63 ...,DR1 ME2 ID1 TK1 CD74 NT5C3B CD320 CD55 SF3A3 E...,GAPDH MT-CYB ACTB SPARC VIM UBB B2M MT2A CD63 ...,DR1 ME2 ID1 TK1 CD74 NT5C3B CD320 CD55 SF3A3 E...,GAPDH MT-CYB SPARC ACTB VIM UBB B2M MT2A CD63 ...,GAPDH MT-CYB SPARC ACTB VIM UBB FN1 B2M MT2A C...,CD74 NT5C3B CD320 TK1 CD55 SF3A3 EI24 CD96 AP1...,511.354167
2,ACH-000788,5-FU,BEZ-235,synergy,MT-ATP8 MT-CO2 MT-ATP6 MT-CO3 MT-CO1 MT-ND4 MT...,A2058,C1=C(C(=O)NC(=O)N1)F,CC(C)(C#N)C1=CC=C(C=C1)N2C3=C4C=C(C=CC4=NC=C3N...,C1=C(C(=O)NC(=O)N1)F || CC(C)(C#N)C1=CC=C(C=C1...,endothelial cell,...,OP-2500 TP8 CO2 TP6 CO3 CO1 ND4 ND2 ND4L EF1A1...,DR1 ME2 ID1 TK1 CD74 NT5C3B CD320 CD55 SF3A3 E...,GAPDH MT-CYB ACTB SPARC VIM UBB B2M MT2A CD63 ...,DR1 ME2 ID1 TK1 CD74 NT5C3B CD320 CD55 SF3A3 E...,GAPDH MT-CYB ACTB SPARC VIM UBB B2M MT2A CD63 ...,DR1 ME2 ID1 TK1 CD74 NT5C3B CD320 CD55 SF3A3 E...,GAPDH MT-CYB SPARC ACTB VIM UBB B2M MT2A CD63 ...,GAPDH MT-CYB SPARC ACTB VIM UBB FN1 B2M MT2A C...,CD74 NT5C3B CD320 TK1 CD55 SF3A3 EI24 CD96 AP1...,511.354167
3,ACH-000788,5-FU,BORTEZOMIB,antagonism,MT-ATP8 MT-CO2 MT-ATP6 MT-CO3 MT-CO1 MT-ND4 MT...,A2058,C1=C(C(=O)NC(=O)N1)F,B(C(CC(C)C)NC(=O)C(CC1=CC=CC=C1)NC(=O)C2=NC=CN...,C1=C(C(=O)NC(=O)N1)F || B(C(CC(C)C)NC(=O)C(CC1...,endothelial cell,...,OP-2500 TP8 CO2 TP6 CO3 CO1 ND4 ND2 ND4L EF1A1...,DR1 ME2 ID1 TK1 CD74 NT5C3B CD320 CD55 SF3A3 E...,GAPDH MT-CYB ACTB SPARC VIM UBB B2M MT2A CD63 ...,DR1 ME2 ID1 TK1 CD74 NT5C3B CD320 CD55 SF3A3 E...,GAPDH MT-CYB ACTB SPARC VIM UBB B2M MT2A CD63 ...,DR1 ME2 ID1 TK1 CD74 NT5C3B CD320 CD55 SF3A3 E...,GAPDH MT-CYB SPARC ACTB VIM UBB B2M MT2A CD63 ...,GAPDH MT-CYB SPARC ACTB VIM UBB FN1 B2M MT2A C...,CD74 NT5C3B CD320 TK1 CD55 SF3A3 EI24 CD96 AP1...,511.354167
4,ACH-000788,5-FU,DASATINIB,synergy,MT-ATP8 MT-CO2 MT-ATP6 MT-CO3 MT-CO1 MT-ND4 MT...,A2058,C1=C(C(=O)NC(=O)N1)F,CC1=C(C(=CC=C1)Cl)NC(=O)C2=CN=C(S2)NC3=NC(=NC(...,C1=C(C(=O)NC(=O)N1)F || CC1=C(C(=CC=C1)Cl)NC(=...,endothelial cell,...,OP-2500 TP8 CO2 TP6 CO3 CO1 ND4 ND2 ND4L EF1A1...,DR1 ME2 ID1 TK1 CD74 NT5C3B CD320 CD55 SF3A3 E...,GAPDH MT-CYB ACTB SPARC VIM UBB B2M MT2A CD63 ...,DR1 ME2 ID1 TK1 CD74 NT5C3B CD320 CD55 SF3A3 E...,GAPDH MT-CYB ACTB SPARC VIM UBB B2M MT2A CD63 ...,DR1 ME2 ID1 TK1 CD74 NT5C3B CD320 CD55 SF3A3 E...,GAPDH MT-CYB SPARC ACTB VIM UBB B2M MT2A CD63 ...,GAPDH MT-CYB SPARC ACTB VIM UBB FN1 B2M MT2A C...,CD74 NT5C3B CD320 TK1 CD55 SF3A3 EI24 CD96 AP1...,511.354167



File saved as: /home/aibox/df_with_c2s_processed.csv.gz


In [6]:
import pandas as pd

path_in = "df_with_c2s.csv.gz"

cols_to_drop = [
    "c2s_cell_type",
    "c2s_treated_d1_sentence",
    "c2s_up_d1",
    "c2s_down_d1",
    "c2s_treated_d2_sentence",
    "c2s_up_d2",
    "c2s_down_d2",
    "c2s_treated_d12_additive",
    "c2s_up_d12_add",
    "c2s_down_d12_add",
    "c2s_treated_d12_direct",
    "c2s_up_synergy",
    "c2s_down_synergy",
    "c2s_synergy_score",
]

df = pd.read_csv(path_in, compression="gzip")

# گزارش اینکه کدوم ستون‌ها واقعاً وجود داشتن
existing = [c for c in cols_to_drop if c in df.columns]
missing  = [c for c in cols_to_drop if c not in df.columns]

df = df.drop(columns=cols_to_drop, errors="ignore")

print("Dropped columns:", existing)
print("Missing columns (not found):", missing)
print("Remaining columns:", df.shape[1], "| Remaining rows:", df.shape[0])

# ذخیره خروجی (اختیاری)
path_out = "df_without_c2s_cols.csv.gz"
df.to_csv(path_out, index=False, compression="gzip")
print("Saved to:", path_out)


Dropped columns: []
Missing columns (not found): ['c2s_cell_type', 'c2s_treated_d1_sentence', 'c2s_up_d1', 'c2s_down_d1', 'c2s_treated_d2_sentence', 'c2s_up_d2', 'c2s_down_d2', 'c2s_treated_d12_additive', 'c2s_up_d12_add', 'c2s_down_d12_add', 'c2s_treated_d12_direct', 'c2s_up_synergy', 'c2s_down_synergy', 'c2s_synergy_score']
Remaining columns: 9 | Remaining rows: 4398
Saved to: df_without_c2s_cols.csv.gz


In [7]:
df = pd.read_csv("df_without_c2s_cols.csv.gz", compression="gzip")
display(df.head(5))


,ModelID,drug1_name,drug2_name,label,cell_sentence,cell,drug1_smiles,drug2_smiles,drugpair_smiles
0,ACH-000788,5-FU,ABT-888,synergy,MT-ATP8 MT-CO2 MT-ATP6 MT-CO3 MT-CO1 MT-ND4 MT...,A2058,C1=C(C(=O)NC(=O)N1)F,CC1(CCCN1)C2=NC3=C(C=CC=C3N2)C(=O)N,C1=C(C(=O)NC(=O)N1)F || CC1(CCCN1)C2=NC3=C(C=C...
1,ACH-000788,5-FU,AZD1775,synergy,MT-ATP8 MT-CO2 MT-ATP6 MT-CO3 MT-CO1 MT-ND4 MT...,A2058,C1=C(C(=O)NC(=O)N1)F,CC(C)(C1=NC(=CC=C1)N2C3=NC(=NC=C3C(=O)N2CC=C)N...,C1=C(C(=O)NC(=O)N1)F || CC(C)(C1=NC(=CC=C1)N2C...
2,ACH-000788,5-FU,BEZ-235,synergy,MT-ATP8 MT-CO2 MT-ATP6 MT-CO3 MT-CO1 MT-ND4 MT...,A2058,C1=C(C(=O)NC(=O)N1)F,CC(C)(C#N)C1=CC=C(C=C1)N2C3=C4C=C(C=CC4=NC=C3N...,C1=C(C(=O)NC(=O)N1)F || CC(C)(C#N)C1=CC=C(C=C1...
3,ACH-000788,5-FU,BORTEZOMIB,antagonism,MT-ATP8 MT-CO2 MT-ATP6 MT-CO3 MT-CO1 MT-ND4 MT...,A2058,C1=C(C(=O)NC(=O)N1)F,B(C(CC(C)C)NC(=O)C(CC1=CC=CC=C1)NC(=O)C2=NC=CN...,C1=C(C(=O)NC(=O)N1)F || B(C(CC(C)C)NC(=O)C(CC1...
4,ACH-000788,5-FU,DASATINIB,synergy,MT-ATP8 MT-CO2 MT-ATP6 MT-CO3 MT-CO1 MT-ND4 MT...,A2058,C1=C(C(=O)NC(=O)N1)F,CC1=C(C(=CC=C1)Cl)NC(=O)C2=CN=C(S2)NC3=NC(=NC(...,C1=C(C(=O)NC(=O)N1)F || CC1=C(C(=CC=C1)Cl)NC(=...


In [17]:
import pandas as pd

# خواندن فایل
path = "df_with_c2s.csv.gz"
df = pd.read_csv(path, compression="gzip")

# تعداد سلول‌های یکتا
unique_cells = df['cell'].nunique()
print(f"🔬 تعداد سلول‌های یکتا: {unique_cells}")

# لیست سلول‌ها
print("\n" + "="*60)
print("لیست سلول‌های یکتا:")
print("="*60)
cell_list = sorted(df['cell'].unique())
for i, cell in enumerate(cell_list, 1):
    count = (df['cell'] == cell).sum()
    print(f"{i:2d}. {cell:20s} ({count:,} رکورد)")

# آمار کلی
print("\n" + "="*60)
print("آمار کلی:")
print("="*60)
print(f"📊 تعداد کل رکوردها: {len(df):,}")
print(f"🔬 تعداد سلول‌های یکتا: {unique_cells}")
print(f"📈 میانگین رکورد به ازای هر سلول: {len(df)/unique_cells:.1f}")

# توزیع تعداد رکوردها
print("\n" + "="*60)
print("توزیع تعداد رکوردها:")
print("="*60)
cell_counts = df['cell'].value_counts()
display(cell_counts.describe())

# سلول‌های با بیشترین و کمترین رکورد
print("\n🔝 5 سلول با بیشترین رکورد:")
display(cell_counts.head(5))

print("\n🔻 5 سلول با کمترین رکورد:")
display(cell_counts.tail(5))


🔬 تعداد سلول‌های یکتا: 52

لیست سلول‌های یکتا:
 1. 786-0                (75 رکورد)
 2. A2058                (428 رکورد)
 3. A2780                (69 رکورد)
 4. A375                 (27 رکورد)
 5. A427                 (24 رکورد)
 6. A498                 (101 رکورد)
 7. A549                 (74 رکورد)
 8. ACHN                 (68 رکورد)
 9. BT-549               (106 رکورد)
10. CAKI-1               (109 رکورد)
11. CAOV3                (2 رکورد)
12. DLD1                 (5 رکورد)
13. DU-145               (84 رکورد)
14. EKVX                 (96 رکورد)
15. ES2                  (2 رکورد)
16. HCT-15               (94 رکورد)
17. HCT116               (112 رکورد)
18. HOP-62               (80 رکورد)
19. HOP-92               (68 رکورد)
20. HS 578T              (75 رکورد)
21. HT144                (1 رکورد)
22. HT29                 (78 رکورد)
23. IGROV1               (81 رکورد)
24. K-562                (95 رکورد)
25. KM12                 (97 رکورد)
26. LOX IMVI             (91 رکورد)
27. MALME-3M    

count     52.000000
mean      84.576923
std       56.047020
min        1.000000
25%       73.750000
50%       83.000000
75%       95.250000
max      428.000000
Name: count, dtype: float64


🔝 5 سلول با بیشترین رکورد:


cell
A2058        428
SF-539       116
RPMI-8226    113
HCT116       112
CAKI-1       109
Name: count, dtype: int64


🔻 5 سلول با کمترین رکورد:


cell
A427     24
DLD1      5
CAOV3     2
ES2       2
HT144     1
Name: count, dtype: int64

In [ ]:
import pandas as pd

path = "c2s27b_out_0_4_in300_out300.csv"
df_out = pd.read_csv(path)

# تعداد سلول‌های یکتا
unique_cells = df_out['cell'].nunique()
print(f"🔬 تعداد سلول‌های یکتا: {unique_cells}")

# لیست سلول‌ها
print("\n" + "="*60)
print("لیست سلول‌های یکتا:")
print("="*60)
cell_list = sorted(df_out['cell'].unique())
for i, cell in enumerate(cell_list, 1):
    count = (df_out['cell'] == cell).sum()
    print(f"{i:2d}. {cell:20s} ({count:,} رکورد)")

# آمار کلی
print("\n" + "="*60)
print("آمار توزیع:")
print("="*60)
print(df_out['cell'].value_counts().describe())

# نمودار توزیع (اختیاری)
print("\n📊 10 سلول با بیشترین رکورد:")
print(df_out['cell'].value_counts().head(10))


In [5]:
import sys
import os, re, gc, warnings, hashlib
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import torch
!{sys.executable} -m pip install -U pip
!{sys.executable} -m pip install -U transformers accelerate sentencepiece
import torch


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 53.7 MB/s  0:00:00
  Attempting uninstall: pip
    Found existing installation: pip 25.3
    Uninstalling pip-25.3:
      Successfully uninstalled pip-25.3
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.3/10.3 MB 185.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 553.3/553.3 kB 35.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 37.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.6/803.6 kB 71.8 MB/s  0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.2.3
    Uninstalling huggingface_hub-1.2.3:
      Successfully uninstalled huggingface_hub-1.2.3━━━━━━━━━━━━━━ 2/6 [huggingface-hub]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6/6 [transformers] [transformers]ub]


In [6]:
import sys, importlib, os, site, subprocess

# 1) نشان بده همین کرنله کجاست
print("python:", sys.executable)
subprocess.check_call([sys.executable, "-m", "pip", "-V"])

# 2) اگر قبلاً چیزی نصفه‌نیمه نصب شده پاکش کن
subprocess.call([sys.executable, "-m", "pip", "uninstall", "-y", "bitsandbytes", "bitsandbytes-cuda110", "bitsandbytes-cuda111",
                 "bitsandbytes-cuda112", "bitsandbytes-cuda113", "bitsandbytes-cuda114", "bitsandbytes-cuda115", "bitsandbytes-cuda116",
                 "bitsandbytes-cuda117", "bitsandbytes-cuda118", "bitsandbytes-cuda120", "bitsandbytes-cuda121", "bitsandbytes-cuda122"])

# 3) نصب دوباره (بدون کش)
subprocess.check_call([sys.executable, "-m", "pip", "install", "-U", "--no-cache-dir", "bitsandbytes"])

# 4) تستِ متادیتا (همون چیزی که Transformers می‌خواست)
import importlib.metadata as md
print("bitsandbytes version:", md.version("bitsandbytes"))

# 5) تست import و مسیر فایل
import bitsandbytes as bnb
print("bitsandbytes file:", bnb.__file__)


python: /venv/main/bin/python
pip 26.0.1 from /venv/main/lib/python3.12/site-packages/pip (python 3.12)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 73.6 MB/s  0:00:00 eta 0:00:01
bitsandbytes version: 0.49.1


bitsandbytes file: /venv/main/lib/python3.12/site-packages/bitsandbytes/__init__.py


In [7]:
import bitsandbytes as bnb
print(bnb.__file__)


/venv/main/lib/python3.12/site-packages/bitsandbytes/__init__.py


In [8]:
import sys, subprocess, torch

print("python:", sys.version)
print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
print("torch cuda version:", torch.version.cuda)
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))

# دیباگ رسمی bitsandbytes
subprocess.run([sys.executable, "-m", "pip", "show", "bitsandbytes"], check=False)
subprocess.run([sys.executable, "-m", "bitsandbytes"], check=False)


python: 3.12.12 | packaged by conda-forge | (main, Oct 22 2025, 23:25:55) [GCC 14.3.0]
torch: 2.10.0+cu130
cuda available: True
torch cuda version: 13.0
gpu: NVIDIA GeForce RTX 5090
Name: bitsandbytes
Version: 0.49.1
Summary: k-bit optimizers and matrix multiplication routines.
Home-page: https://github.com/bitsandbytes-foundation/bitsandbytes
Author: 
Author-email: Tim Dettmers <dettmers@cs.washington.edu>
License-Expression: MIT
Location: /venv/main/lib/python3.12/site-packages
Requires: numpy, packaging, torch
Required-by: 
=================== bitsandbytes v0.49.1 ===================
Platform: Linux-6.8.0-58-lowlatency-x86_64-with-glibc2.39
  libc: glibc-2.39
Python: 3.12.12
PyTorch: 2.10.0+cu130
  CUDA: 13.0
  HIP: N/A
  XPU: N/A
Related packages:
  accelerate: 1.12.0
  diffusers: not found
  numpy: 2.4.1
  pip: 26.0.1
  peft: not found
  safetensors: 0.7.0
  transformers: 5.1.0
  triton: 3.6.0
  trl: not found
PyTorch settings found: CUDA_VERSION=130, Highest Compute Capability: (

CompletedProcess(args=['/venv/main/bin/python', '-m', 'bitsandbytes'], returncode=0)

In [25]:
# # ============================================================
# # C2S-Scale Gemma-2 27B (vandijklab) — rows 0..100
# # K_IN=300, K_OUT=300
# # - Finds existing local HF snapshot (no re-download)
# # - Forces HF_HUB_CACHE to the found cache
# # - Loads 4-bit (bitsandbytes) + device_map=auto
# # - Runs rows 0..100 and saves CSV (resume-safe)
# # ============================================================

# import os, re, gc, sys, subprocess, importlib.util, warnings, glob, hashlib
# from pathlib import Path

# import numpy as np
# import pandas as pd
# from tqdm.auto import tqdm

# # ----------------------------
# # 0) Ensure packages
# # ----------------------------
# def _pip_install(pkgs):
#     subprocess.check_call([sys.executable, "-m", "pip", "install", "-U", *pkgs])

# need = []
# for p in ["transformers", "accelerate", "bitsandbytes", "sentencepiece", "protobuf"]:
#     if importlib.util.find_spec(p) is None:
#         need.append(p)

# if need:
#     _pip_install(["pip"])
#     _pip_install(["transformers<5", "accelerate", "bitsandbytes", "sentencepiece", "protobuf"])

# import torch
# from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
# from transformers import logging as hf_logging
# hf_logging.set_verbosity_error()

# # ----------------------------
# # 1) Settings
# # ----------------------------
# MODEL_ID   = "vandijklab/C2S-Scale-Gemma-2-27B"
# INPUT_PATH = "df_with_c2s.csv.gz"

# START_ROW  = 0
# END_ROW    = 5  # inclusive
# CHUNK_SIZE = 5

# K_IN  = 300
# K_OUT = 300
# TOPN  = 10

# OUTPUT_CSV = f"c2s27b_out_{START_ROW}_{END_ROW}_in{K_IN}_out{K_OUT}.csv"

# print("MODEL:", MODEL_ID)
# print("INPUT:", INPUT_PATH)
# print("OUTPUT:", OUTPUT_CSV)
# print(f"ROWS: {START_ROW}..{END_ROW} | K_IN={K_IN} K_OUT={K_OUT}")

# # ----------------------------
# # 2) Find local HF snapshot (so no download needed)
# # ----------------------------
# def _expand(p: str) -> str:
#     return str(Path(p).expanduser().resolve())

# def find_local_snapshot(repo_id: str, extra_roots=None):
#     """
#     Search common HF hub cache roots for:
#       <hub_root>/models--ORG--REPO/snapshots/<hash>/
#     Return: (snapshot_path, hub_root) or (None, None)
#     """
#     repo_dirname = "models--" + repo_id.replace("/", "--")

#     roots = []
#     # Highest priority: explicit env
#     for k in ["HF_HUB_CACHE", "HUGGINGFACE_HUB_CACHE"]:
#         v = os.environ.get(k)
#         if v:
#             roots.append(_expand(v))
#     # If HF_HOME is set, hub cache defaults to HF_HOME/hub
#     hf_home = os.environ.get("HF_HOME")
#     if hf_home:
#         roots.append(_expand(os.path.join(hf_home, "hub")))

#     # Common defaults (per HF docs: ~/.cache/huggingface/hub)
#     roots += [
#         _expand("~/.cache/huggingface/hub"),
#         "/root/.cache/huggingface/hub",
#         "/workspace/hf_home/hub",
#         "/workspace/huggingface/hub",
#         "/workspace/hub",
#         "/workspace/.cache/huggingface/hub",
#         "/mnt/.cache/huggingface/hub",
#         "/tmp/.cache/huggingface/hub",
#     ]

#     if extra_roots:
#         roots += [ _expand(r) for r in extra_roots ]

#     # De-dup while preserving order
#     seen = set()
#     roots = [r for r in roots if not (r in seen or seen.add(r))]

#     candidates = []
#     for hub_root in roots:
#         p = Path(hub_root) / repo_dirname / "snapshots"
#         if p.is_dir():
#             snaps = sorted(p.glob("*"), key=lambda x: x.stat().st_mtime, reverse=True)
#             for s in snaps:
#                 # minimal sanity: config + tokenizer.model or tokenizer.json exist
#                 if (s / "config.json").exists() and (
#                     (s / "tokenizer.model").exists() or (s / "tokenizer.json").exists()
#                 ):
#                     candidates.append((s, Path(hub_root)))
#             if candidates:
#                 # return newest valid snapshot in this root
#                 return str(candidates[0][0]), str(candidates[0][1])
#     return None, None

# # اگر می‌خوای دستی هم اضافه کنی (مثلاً یک mount خاص):
# EXTRA_CACHE_ROOTS = []  # e.g. ["/data/hf_cache", "/ssd/hf_cache"]

# LOCAL_SNAPSHOT, HUB_ROOT = find_local_snapshot(MODEL_ID, EXTRA_CACHE_ROOTS)

# if LOCAL_SNAPSHOT:
#     print("✅ Found local snapshot:")
#     print("   SNAPSHOT:", LOCAL_SNAPSHOT)
#     print("   HUB_ROOT:", HUB_ROOT)

#     # Make sure transformers/hub read from the correct cache root
#     os.environ["HF_HUB_CACHE"] = HUB_ROOT  # highest priority for hub cache :contentReference[oaicite:1]{index=1}
#     os.environ["HF_HUB_OFFLINE"] = "1"     # force offline (no HTTP) :contentReference[oaicite:2]{index=2}
#     LOCAL_FILES_ONLY = True
#     LOAD_REF = LOCAL_SNAPSHOT
# else:
#     # اگر واقعاً پیدا نشد، باید دانلود کنی (در این حالت HF_HUB_OFFLINE نباید 1 باشد)
#     print("⚠️ Local snapshot not found in common cache roots.")
#     print("   Either the model was removed, or it lives in a different cache path.")
#     print("   If you DO have internet and space, disable offline and allow download.")
#     os.environ.pop("HF_HUB_OFFLINE", None)
#     LOCAL_FILES_ONLY = False
#     LOAD_REF = MODEL_ID

# # ----------------------------
# # 3) Robust df loading
# # ----------------------------
# NEEDED_COLS = ["cell","drug1_name","drug1_smiles","drug2_name","drug2_smiles","label","cell_sentence"]

# def read_csv_robust(path):
#     if not os.path.exists(path):
#         raise FileNotFoundError(f"File not found: {path}")

#     def _usecols(c): return c in set(NEEDED_COLS)

#     encs = ["utf-8-sig", "utf-8", "cp1252", "latin1"]
#     last_err = None
#     for enc in encs:
#         try:
#             df = pd.read_csv(
#                 path,
#                 compression="infer",
#                 encoding=enc,
#                 encoding_errors="replace",
#                 low_memory=False,
#                 usecols=_usecols
#             )
#             return df, enc
#         except Exception as e:
#             last_err = e
#     raise last_err

# df, used_enc = read_csv_robust(INPUT_PATH)
# print("✅ df loaded. shape:", df.shape, "| encoding:", used_enc)
# print("columns:", list(df.columns))

# # ----------------------------
# # 4) Gene sentence helpers
# # ----------------------------
# GENE_TOKEN_RE = re.compile(r"^(MT-[A-Z0-9]+|HLA-[A-Z0-9]+|[A-Z]{2,}[A-Z0-9\-]*\d*[A-Z0-9\-]*)$")

# def sanitize_sentence(s: str, limit: int | None = None) -> str:
#     if s is None or (isinstance(s, float) and np.isnan(s)):
#         return ""
#     s = str(s).upper()
#     s = re.sub(r"[;,\.\?\!\t\|/]+", " ", s)
#     s = re.sub(r"[^A-Z0-9\-\s]+", " ", s)
#     tokens = [t for t in s.split() if GENE_TOKEN_RE.match(t)]
#     seen, out = set(), []
#     for t in tokens:
#         if t not in seen:
#             seen.add(t)
#             out.append(t)
#     if limit is not None:
#         out = out[:limit]
#     return " ".join(out)

# def ensure_exact_k_genes(treated_sentence: str, baseline_sentence: str, k: int) -> str:
#     treated = str(treated_sentence).split() if treated_sentence else []
#     base    = str(baseline_sentence).split() if baseline_sentence else []
#     exist = set(treated)
#     if len(treated) < k:
#         for g in base:
#             if g not in exist:
#                 treated.append(g)
#                 exist.add(g)
#             if len(treated) >= k:
#                 break
#     return " ".join(treated[:k])

# def rank_map(sentence: str) -> dict:
#     return {g: i for i, g in enumerate(str(sentence).split())}

# def top_changed_genes(untreated: str, treated: str, topN: int):
#     r0, r1 = rank_map(untreated), rank_map(treated)
#     shared = set(r0).intersection(r1)
#     deltas = [(g, r0[g] - r1[g]) for g in shared]  # + => moved up
#     up   = [g for g,_ in sorted(deltas, key=lambda x: -x[1])[:topN]]
#     down = [g for g,_ in sorted(deltas, key=lambda x:  x[1])[:topN]]
#     return " ".join(up), " ".join(down)

# def additive_combo(untreated: str, t1: str, t2: str, K_out: int):
#     r0, r1, r2 = rank_map(untreated), rank_map(t1), rank_map(t2)
#     BIG = len(r0) + 500
#     genes = list({*r0.keys(), *r1.keys(), *r2.keys()})
#     rows = []
#     for g in genes:
#         b  = r0.get(g, BIG)
#         d1 = b - r1.get(g, BIG)
#         d2 = b - r2.get(g, BIG)
#         rows.append((g, b - (d1 + d2)))
#     rows.sort(key=lambda x: x[1])
#     return " ".join([g for g,_ in rows[:K_out]])

# def synergy_from_direct_vs_additive(untreated, treated_add, treated_direct, topN):
#     r0, ra, rd = rank_map(untreated), rank_map(treated_add), rank_map(treated_direct)
#     genes = list(set(ra).intersection(rd).intersection(r0))
#     diffs = []
#     for g in genes:
#         diff = (r0[g] - rd[g]) - (r0[g] - ra[g])  # direct - additive
#         diffs.append((g, diff))
#     if not diffs:
#         return "", "", 0.0
#     diffs.sort(key=lambda x: -x[1])
#     upS = [g for g,_ in diffs[:topN]]
#     diffs.sort(key=lambda x: x[1])
#     downS = [g for g,_ in diffs[:topN]]
#     score = float(np.mean([abs(d) for _,d in diffs[:min(100, len(diffs))]]))
#     return " ".join(upS), " ".join(downS), score

# # ----------------------------
# # 5) Resume helpers
# # ----------------------------
# def load_done_indices_csv(path: str, start: int, end: int) -> set:
#     if not os.path.exists(path):
#         return set()
#     done = set()
#     for chunk in pd.read_csv(path, chunksize=50_000, usecols=["orig_index"], engine="python", on_bad_lines="skip"):
#         for ix in chunk["orig_index"].astype(int).tolist():
#             if start <= ix <= end:
#                 done.add(ix)
#     return done

# # ----------------------------
# # 6) Load model (4-bit)
# # ----------------------------
# def bf16_supported():
#     try:
#         return torch.cuda.is_available() and torch.cuda.is_bf16_supported()
#     except Exception:
#         return False

# compute_dtype = torch.bfloat16 if bf16_supported() else (torch.float16 if torch.cuda.is_available() else torch.float32)

# bnb_config = BitsAndBytesConfig(
#     load_in_4bit=True,
#     bnb_4bit_quant_type="nf4",
#     bnb_4bit_use_double_quant=True,
#     bnb_4bit_compute_dtype=compute_dtype,
# )

# tokenizer = AutoTokenizer.from_pretrained(
#     LOAD_REF,
#     local_files_only=LOCAL_FILES_ONLY,
# )
# if tokenizer.pad_token is None:
#     tokenizer.pad_token = tokenizer.eos_token

# # memory hints (optional)
# max_memory = None
# if torch.cuda.is_available():
#     # adjust if needed
#     max_memory = {0: "28GiB", "cpu": "96GiB"}
#     print("max_memory:", max_memory)

# model = AutoModelForCausalLM.from_pretrained(
#     LOAD_REF,
#     device_map="auto",
#     quantization_config=bnb_config,
#     torch_dtype=compute_dtype,
#     low_cpu_mem_usage=True,
#     max_memory=max_memory,
#     local_files_only=LOCAL_FILES_ONLY,
#     offload_folder="offload_27b",
#     offload_state_dict=True,
# )
# model.eval()
# model.config.pad_token_id = tokenizer.pad_token_id

# MAX_CTX = int(getattr(model.config, "max_position_embeddings", 8192))

# def get_input_device(m):
#     try:
#         return m.get_input_embeddings().weight.device
#     except Exception:
#         return next(m.parameters()).device

# @torch.inference_mode()
# def generate_after_marker(prompt: str, marker: str, max_new_tokens: int):
#     reserve = max_new_tokens + 16
#     max_in = max(64, MAX_CTX - reserve)

#     inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=max_in)
#     dev = get_input_device(model)
#     inputs = {k: v.to(dev) for k, v in inputs.items()}

#     out = model.generate(
#         **inputs,
#         max_new_tokens=max_new_tokens,
#         do_sample=False,
#         pad_token_id=tokenizer.pad_token_id,
#         eos_token_id=tokenizer.eos_token_id,
#         use_cache=True,
#     )
#     txt = tokenizer.decode(out[0], skip_special_tokens=True)
#     return txt.split(marker, 1)[1].strip() if marker in txt else txt.strip()

# def clean_cell_type(txt: str) -> str:
#     txt = re.sub(r'[\n\r\t]+', ' ', str(txt))
#     txt = re.sub(r'\s+', ' ', txt).strip(" .:'\"`")
#     return txt.strip()

# # ----------------------------
# # 7) Prompts
# # ----------------------------
# CELLTYPE_MARKER = "The cell type corresponding to these genes is:"
# PERT_MARKER = "A:"

# def prompt_celltype(cell_sentence: str, organism="Homo sapiens"):
#     num_genes = len(cell_sentence.split())
#     return (
#         f"The following is a list of {num_genes} gene names ordered by descending expression level in a {organism} cell. "
#         f"Your task is to give the cell type which this cell belongs to based on its gene expression.\n"
#         f"Cell sentence: {cell_sentence}.\n"
#         f"{CELLTYPE_MARKER}"
#     )

# def prompt_treated_single(cell_line: str, baseline: str, drug: str, K_in: int, K_out: int):
#     base_trim = " ".join(baseline.split()[:K_in])
#     return (
#         "Q: Predict the treated cell sentence after perturbation.\n"
#         "Context:\n"
#         "- Organism: Homo sapiens\n"
#         f"- Cell line: {cell_line}\n"
#         f"- Baseline (untreated ranked genes, top-{K_in}): {base_trim}\n"
#         f"- Perturbation: drug = {drug}; time = 24h\n"
#         f"Instruction: Return ONLY {K_out} human gene symbols, space-separated, highest rank first. No explanation.\n"
#         f"{PERT_MARKER}"
#     )

# def prompt_treated_combo(cell_line: str, baseline: str, d1: str, d2: str, K_in: int, K_out: int):
#     base_trim = " ".join(baseline.split()[:K_in])
#     return (
#         "Q: Predict the treated cell sentence after combined perturbation.\n"
#         "Context:\n"
#         "- Organism: Homo sapiens\n"
#         f"- Cell line: {cell_line}\n"
#         f"- Baseline (untreated ranked genes, top-{K_in}): {base_trim}\n"
#         f"- Perturbation: drugs = [{d1}, {d2}]; time = 24h\n"
#         f"Instruction: Return ONLY {K_out} human gene symbols, space-separated, highest rank first. No explanation.\n"
#         f"{PERT_MARKER}"
#     )

# def predict_celltype_27b(baseline_sentence: str) -> str:
#     p = prompt_celltype(baseline_sentence)
#     ans = generate_after_marker(p, CELLTYPE_MARKER, max_new_tokens=24)
#     return clean_cell_type(ans)

# def predict_treated_27b(cell_line: str, baseline: str, drug: str, K_in: int, K_out: int) -> str:
#     p = prompt_treated_single(cell_line, baseline, drug, K_in, K_out)
#     raw = generate_after_marker(p, PERT_MARKER, max_new_tokens=K_out + 128)
#     genes = sanitize_sentence(raw, limit=None)
#     return ensure_exact_k_genes(genes, baseline, K_out)

# def predict_combo_direct_27b(cell_line: str, baseline: str, d1: str, d2: str, K_in: int, K_out: int) -> str:
#     p = prompt_treated_combo(cell_line, baseline, d1, d2, K_in, K_out)
#     raw = generate_after_marker(p, PERT_MARKER, max_new_tokens=K_out + 128)
#     genes = sanitize_sentence(raw, limit=None)
#     return ensure_exact_k_genes(genes, baseline, K_out)

# # ----------------------------
# # 8) Main loop: rows 0..100
# # ----------------------------
# end_eff = min(END_ROW, len(df) - 1)
# target_indices = list(range(START_ROW, end_eff + 1))

# done = load_done_indices_csv(OUTPUT_CSV, START_ROW, end_eff)
# todo = [ix for ix in target_indices if ix not in done]

# print(f"✅ target rows: {len(target_indices)} | done: {len(done)} | remaining: {len(todo)}")

# if not todo:
#     print("🎉 Already finished rows 0..100 (based on OUTPUT_CSV).")
# else:
#     file_exists = os.path.exists(OUTPUT_CSV)
#     buffer = []
#     topN_eff = int(min(max(1, TOPN), max(1, K_OUT)))

#     cache_ct = {}
#     cache_single = {}

#     for ix in tqdm(todo, desc=f"C2S27B rows {START_ROW}-{end_eff}"):
#         row = df.iloc[ix]

#         cell_name = str(row.get("cell", ""))
#         d1_name   = str(row.get("drug1_name", ""))
#         d2_name   = str(row.get("drug2_name", ""))

#         baseline = sanitize_sentence(row.get("cell_sentence", ""), limit=K_IN)
#         sent_hash = hashlib.md5(baseline.encode("utf-8")).hexdigest()

#         # cell type
#         if sent_hash in cache_ct:
#             ct = cache_ct[sent_hash]
#         else:
#             try:
#                 ct = predict_celltype_27b(baseline) if baseline else ""
#             except Exception as e:
#                 warnings.warn(f"cell-type failed idx={ix}: {e}")
#                 ct = ""
#             cache_ct[sent_hash] = ct

#         # drug1
#         key1 = (sent_hash, d1_name)
#         if key1 in cache_single:
#             t1, up1, dn1 = cache_single[key1]
#         else:
#             try:
#                 t1 = predict_treated_27b(cell_name, baseline, d1_name, K_in=K_IN, K_out=K_OUT) if d1_name else ""
#             except Exception as e:
#                 warnings.warn(f"drug1 failed idx={ix}: {e}")
#                 t1 = ""
#             up1, dn1 = top_changed_genes(baseline, t1, topN=topN_eff) if t1 else ("", "")
#             cache_single[key1] = (t1, up1, dn1)

#         # drug2
#         key2 = (sent_hash, d2_name)
#         if key2 in cache_single:
#             t2, up2, dn2 = cache_single[key2]
#         else:
#             try:
#                 t2 = predict_treated_27b(cell_name, baseline, d2_name, K_in=K_IN, K_out=K_OUT) if d2_name else ""
#             except Exception as e:
#                 warnings.warn(f"drug2 failed idx={ix}: {e}")
#                 t2 = ""
#             up2, dn2 = top_changed_genes(baseline, t2, topN=topN_eff) if t2 else ("", "")
#             cache_single[key2] = (t2, up2, dn2)

#         # additive combo (no model)
#         t12_add = additive_combo(baseline, t1, t2, K_out=K_OUT) if (t1 and t2) else ""
#         upA, dnA = top_changed_genes(baseline, t12_add, topN=topN_eff) if t12_add else ("", "")

#         # direct combo + synergy
#         try:
#             t12_dir = predict_combo_direct_27b(cell_name, baseline, d1_name, d2_name, K_in=K_IN, K_out=K_OUT) if (d1_name and d2_name) else ""
#         except Exception as e:
#             warnings.warn(f"combo direct failed idx={ix}: {e}")
#             t12_dir = ""

#         if t12_add and t12_dir:
#             upS, dnS, sc = synergy_from_direct_vs_additive(baseline, t12_add, t12_dir, topN=topN_eff)
#         else:
#             upS, dnS, sc = "", "", 0.0

#         out_row = {
#             "orig_index": ix,
#             "cell": cell_name,
#             "drug1_name": d1_name,
#             "drug1_smiles": row.get("drug1_smiles", ""),
#             "drug2_name": d2_name,
#             "drug2_smiles": row.get("drug2_smiles", ""),
#             "label": row.get("label", np.nan),

#             "cell_sentence": baseline,
#             "c2s_cell_type": ct,

#             "c2s_treated_d1_sentence": t1,
#             "c2s_up_d1": up1,
#             "c2s_down_d1": dn1,

#             "c2s_treated_d2_sentence": t2,
#             "c2s_up_d2": up2,
#             "c2s_down_d2": dn2,

#             "c2s_treated_d12_additive": t12_add,
#             "c2s_up_d12_add": upA,
#             "c2s_down_d12_add": dnA,

#             "c2s_treated_d12_direct": t12_dir,
#             "c2s_up_synergy": upS,
#             "c2s_down_synergy": dnS,
#             "c2s_synergy_score": sc,
#         }

#         buffer.append(out_row)

#         # save chunk
#         if len(buffer) >= CHUNK_SIZE:
#             pd.DataFrame(buffer).to_csv(
#                 OUTPUT_CSV, index=False,
#                 mode=("a" if file_exists else "w"),
#                 header=(not file_exists),
#             )
#             file_exists = True
#             buffer.clear()

#             if torch.cuda.is_available():
#                 torch.cuda.empty_cache()
#             gc.collect()

#     # save remaining
#     if buffer:
#         pd.DataFrame(buffer).to_csv(
#             OUTPUT_CSV, index=False,
#             mode=("a" if file_exists else "w"),
#             header=(not file_exists),
#         )
#         buffer.clear()

#     print("✅ Done. Saved:", OUTPUT_CSV)


In [11]:
df.shape

(4398, 9)

In [1]:
import pandas as pd

# خواندن فایل
path = "df_with_c2s.csv.gz"
df = pd.read_csv(path, compression="gzip")

# اطلاعات کلی
print(f"📊 ابعاد: {df.shape[0]:,} سطر × {df.shape[1]} ستون")
print("\n" + "="*70)
print("ستون‌های موجود:")
print("="*70)
for i, col in enumerate(df.columns, 1):
    dtype = df[col].dtype
    print(f"{i:2d}. {col:40s} ({dtype})")

# نمایش 10 سطر اول
print("\n" + "="*70)
print("10 سطر اول:")
print("="*70)
display(df.head(10))


📊 ابعاد: 4,398 سطر × 9 ستون

ستون‌های موجود:
 1. ModelID                                  (str)
 2. drug1_name                               (str)
 3. drug2_name                               (str)
 4. label                                    (str)
 5. cell_sentence                            (str)
 6. cell                                     (str)
 7. drug1_smiles                             (str)
 8. drug2_smiles                             (str)
 9. drugpair_smiles                          (str)

10 سطر اول:


,ModelID,drug1_name,drug2_name,label,cell_sentence,cell,drug1_smiles,drug2_smiles,drugpair_smiles
0,ACH-000788,5-FU,ABT-888,synergy,MT-ATP8 MT-CO2 MT-ATP6 MT-CO3 MT-CO1 MT-ND4 MT...,A2058,C1=C(C(=O)NC(=O)N1)F,CC1(CCCN1)C2=NC3=C(C=CC=C3N2)C(=O)N,C1=C(C(=O)NC(=O)N1)F || CC1(CCCN1)C2=NC3=C(C=C...
1,ACH-000788,5-FU,AZD1775,synergy,MT-ATP8 MT-CO2 MT-ATP6 MT-CO3 MT-CO1 MT-ND4 MT...,A2058,C1=C(C(=O)NC(=O)N1)F,CC(C)(C1=NC(=CC=C1)N2C3=NC(=NC=C3C(=O)N2CC=C)N...,C1=C(C(=O)NC(=O)N1)F || CC(C)(C1=NC(=CC=C1)N2C...
2,ACH-000788,5-FU,BEZ-235,synergy,MT-ATP8 MT-CO2 MT-ATP6 MT-CO3 MT-CO1 MT-ND4 MT...,A2058,C1=C(C(=O)NC(=O)N1)F,CC(C)(C#N)C1=CC=C(C=C1)N2C3=C4C=C(C=CC4=NC=C3N...,C1=C(C(=O)NC(=O)N1)F || CC(C)(C#N)C1=CC=C(C=C1...
3,ACH-000788,5-FU,BORTEZOMIB,antagonism,MT-ATP8 MT-CO2 MT-ATP6 MT-CO3 MT-CO1 MT-ND4 MT...,A2058,C1=C(C(=O)NC(=O)N1)F,B(C(CC(C)C)NC(=O)C(CC1=CC=CC=C1)NC(=O)C2=NC=CN...,C1=C(C(=O)NC(=O)N1)F || B(C(CC(C)C)NC(=O)C(CC1...
4,ACH-000788,5-FU,DASATINIB,synergy,MT-ATP8 MT-CO2 MT-ATP6 MT-CO3 MT-CO1 MT-ND4 MT...,A2058,C1=C(C(=O)NC(=O)N1)F,CC1=C(C(=CC=C1)Cl)NC(=O)C2=CN=C(S2)NC3=NC(=NC(...,C1=C(C(=O)NC(=O)N1)F || CC1=C(C(=CC=C1)Cl)NC(=...
5,ACH-000788,5-FU,DINACICLIB,synergy,MT-ATP8 MT-CO2 MT-ATP6 MT-CO3 MT-CO1 MT-ND4 MT...,A2058,C1=C(C(=O)NC(=O)N1)F,CCC1=C2N=C(C=C(N2N=C1)NCC3=C[N+](=CC=C3)[O-])N...,C1=C(C(=O)NC(=O)N1)F || CCC1=C2N=C(C=C(N2N=C1)...
6,ACH-000788,5-FU,ERLOTINIB,synergy,MT-ATP8 MT-CO2 MT-ATP6 MT-CO3 MT-CO1 MT-ND4 MT...,A2058,C1=C(C(=O)NC(=O)N1)F,COCCOC1=C(C=C2C(=C1)C(=NC=N2)NC3=CC=CC(=C3)C#C...,C1=C(C(=O)NC(=O)N1)F || COCCOC1=C(C=C2C(=C1)C(...
7,ACH-000788,5-FU,GELDANAMYCIN,synergy,MT-ATP8 MT-CO2 MT-ATP6 MT-CO3 MT-CO1 MT-ND4 MT...,A2058,C1=C(C(=O)NC(=O)N1)F,CC1CC(C(C(C=C(C(C(C=CC=C(C(=O)NC2=CC(=O)C(=C(C...,C1=C(C(=O)NC(=O)N1)F || CC1CC(C(C(C=C(C(C(C=CC...
8,ACH-000788,5-FU,L778123,synergy,MT-ATP8 MT-CO2 MT-ATP6 MT-CO3 MT-CO1 MT-ND4 MT...,A2058,C1=C(C(=O)NC(=O)N1)F,C1CN(C(=O)CN1CC2=CN=CN2CC3=CC=C(C=C3)C#N)C4=CC...,C1=C(C(=O)NC(=O)N1)F || C1CN(C(=O)CN1CC2=CN=CN...
9,ACH-000788,5-FU,LAPATINIB,synergy,MT-ATP8 MT-CO2 MT-ATP6 MT-CO3 MT-CO1 MT-ND4 MT...,A2058,C1=C(C(=O)NC(=O)N1)F,CS(=O)(=O)CCNCC1=CC=C(O1)C2=CC3=C(C=C2)N=CN=C3...,C1=C(C(=O)NC(=O)N1)F || CS(=O)(=O)CCNCC1=CC=C(...


In [ ]:
# ============================================================
# C2S-Scale Gemma-2 27B (vandijklab) — rows 0..100 (resume-safe)
# K_IN=300, K_OUT=300
# - Offline / local cache friendly (اگر قبلا دانلود کرده‌ای)
# - Append CSV + fsync + atomic state.json
# - Resume from last completed row
# ============================================================

import os, re, gc, sys, json, hashlib, warnings, subprocess, importlib.util
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

# ----------------------------
# 0) Install (optional, one time)
# ----------------------------
def _pip_install(pkgs):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-U", *pkgs])

need = []
for p in ["transformers", "accelerate", "bitsandbytes", "sentencepiece", "protobuf"]:
    if importlib.util.find_spec(p) is None:
        need.append(p)

if need:
    _pip_install(["pip"])
    _pip_install(["transformers<5", "accelerate", "bitsandbytes", "sentencepiece", "protobuf"])

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from transformers import logging as hf_logging
hf_logging.set_verbosity_error()

# ----------------------------
# 1) Settings
# ----------------------------
MODEL_ID   = "vandijklab/C2S-Scale-Gemma-2-27B"

INPUT_PATH = "df_with_c2s.csv.gz"
START_ROW  = 0
END_ROW    = 4396    # دقیقاً 0..100
#END_ROW    = 4397     # دقیقاً 0..100

K_IN  = 300
K_OUT = 300

# اگر قبلاً مدل را دانلود کرده‌ای و محیطت آفلاین/محدود است:
LOCAL_FILES_ONLY = False   # کلیدی‌ترین گزینه برای جلوگیری از دانلود

OUTPUT_CSV  = f"c2s27b_out_{START_ROW}_{END_ROW}_in{K_IN}_out{K_OUT}.csv"
STATE_JSON  = OUTPUT_CSV + ".state.json"  # برای ریکاوری سریع‌تر

print("MODEL:", MODEL_ID)
print("INPUT:", INPUT_PATH)
print("OUTPUT:", OUTPUT_CSV)
print("ROWS:", START_ROW, "..", END_ROW, "| K_IN:", K_IN, "K_OUT:", K_OUT)

# ----------------------------
# 2) Robust df loading (only needed columns)
# ----------------------------
NEEDED_COLS = [
    "cell", "drug1_name", "drug2_name", "cell_sentence",
    "drug1_smiles", "drug2_smiles", "label"
]

def read_csv_robust(path, nrows=None):
    if not os.path.exists(path):
        raise FileNotFoundError(f"File not found: {path}")

    def _usecols(c):
        return c in set(NEEDED_COLS)

    encs = ["utf-8-sig", "utf-8", "cp1252", "latin1"]
    last_err = None
    for enc in encs:
        try:
            df = pd.read_csv(
                path,
                compression="infer",
                encoding=enc,
                encoding_errors="replace",
                low_memory=False,
                usecols=_usecols,
                nrows=nrows
            )
            return df, enc
        except Exception as e:
            last_err = e
    raise last_err

# چون فقط 0..100 می‌خوایم، همین‌قدر می‌خونیم (سریع‌تر)
df, used_enc = read_csv_robust(INPUT_PATH, nrows=END_ROW + 1)
print("✅ df loaded. shape:", df.shape, "| encoding used:", used_enc)
print("columns:", list(df.columns))

required = {"cell", "drug1_name", "drug2_name", "cell_sentence"}
missing = required - set(df.columns)
if missing:
    raise ValueError(f"Missing required columns in df: {missing}")

# ----------------------------
# 3) Gene sentence helpers
# ----------------------------
GENE_TOKEN_RE = re.compile(r"^(MT-[A-Z0-9]+|HLA-[A-Z0-9]+|[A-Z]{2,}[A-Z0-9\-]*\d*[A-Z0-9\-]*)$")

def sanitize_sentence(s: str, limit: int | None = None) -> str:
    if s is None or (isinstance(s, float) and np.isnan(s)):
        return ""
    s = str(s).upper()
    s = re.sub(r"[;,\.\?\!\t\|/]+", " ", s)
    s = re.sub(r"[^A-Z0-9\-\s]+", " ", s)
    tokens = [t for t in s.split() if GENE_TOKEN_RE.match(t)]
    seen, out = set(), []
    for t in tokens:
        if t not in seen:
            seen.add(t)
            out.append(t)
    if limit is not None:
        out = out[:limit]
    return " ".join(out)

def ensure_exact_k_genes(treated_sentence: str, baseline_sentence: str, k: int) -> str:
    treated = str(treated_sentence).split() if treated_sentence else []
    base    = str(baseline_sentence).split() if baseline_sentence else []
    exist = set(treated)
    if len(treated) < k:
        for g in base:
            if g not in exist:
                treated.append(g)
                exist.add(g)
            if len(treated) >= k:
                break
    return " ".join(treated[:k])

# ----------------------------
# 4) Resume helpers (CSV + state.json)
# ----------------------------
def atomic_write_json(path: str, obj: dict):
    tmp = path + ".tmp"
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False)
        f.flush()
        os.fsync(f.fileno())
    os.replace(tmp, path)

def read_state(path: str) -> dict:
    if not os.path.exists(path):
        return {}
    try:
        with open(path, "r", encoding="utf-8") as f:
            return json.load(f)
    except Exception:
        return {}

def get_max_orig_index(output_csv: str) -> int | None:
    if not os.path.exists(output_csv):
        return None
    max_ix = None
    try:
        for chunk in pd.read_csv(
            output_csv,
            usecols=["orig_index"],
            chunksize=200_000,
            engine="python",
            on_bad_lines="skip",
        ):
            if len(chunk) == 0:
                continue
            v = int(chunk["orig_index"].max())
            max_ix = v if max_ix is None else max(max_ix, v)
    except Exception:
        # اگر فایل خروجی خراب/نیمه‌کاره باشد، همین None برمی‌گردد و از اول شروع می‌شود
        return None
    return max_ix

def compute_resume_start(start_row: int, end_row: int, output_csv: str, state_json: str) -> int:
    last = -1
    st = read_state(state_json)
    if isinstance(st, dict) and "last_committed" in st:
        try:
            last = max(last, int(st["last_committed"]))
        except Exception:
            pass
    mx = get_max_orig_index(output_csv)
    if mx is not None:
        last = max(last, mx)
    return max(start_row, last + 1)

# ----------------------------
# 5) Load model (4-bit) — local cache friendly
# ----------------------------
def bf16_supported():
    try:
        return torch.cuda.is_available() and torch.cuda.is_bf16_supported()
    except Exception:
        return False

compute_dtype = torch.bfloat16 if bf16_supported() else (torch.float16 if torch.cuda.is_available() else torch.float32)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=compute_dtype,
)

def find_local_snapshot(repo_id: str) -> str | None:
    """
    دنبال snapshot محلی مدل می‌گردد تا بتوانی بدون اینترنت لود کنی.
    """
    candidates = []

    # env overrides
    hf_hub_cache = os.environ.get("HF_HUB_CACHE")
    hf_home = os.environ.get("HF_HOME")
    if hf_hub_cache:
        candidates.append(hf_hub_cache)
    if hf_home:
        candidates.append(os.path.join(hf_home, "hub"))

    # common defaults
    candidates += [
        os.path.expanduser("~/.cache/huggingface/hub"),
        "/root/.cache/huggingface/hub",
        "/workspace/hf_home/hub",
        "/workspace/hf_hub_cache",
        "/workspace/hf_cache/hub",
    ]

    repo_dirname = "models--" + repo_id.replace("/", "--")
    for hub in candidates:
        if not hub:
            continue
        repo_dir = os.path.join(hub, repo_dirname)
        snaps = os.path.join(repo_dir, "snapshots")
        if os.path.isdir(snaps):
            subdirs = [os.path.join(snaps, d) for d in os.listdir(snaps)]
            subdirs = [d for d in subdirs if os.path.isdir(d)]
            if subdirs:
                # جدیدترین snapshot
                subdirs.sort(key=lambda p: os.path.getmtime(p), reverse=True)
                return subdirs[0]
    return None

LOCAL_SNAPSHOT = find_local_snapshot(MODEL_ID)
LOAD_REF = LOCAL_SNAPSHOT if (LOCAL_FILES_ONLY and LOCAL_SNAPSHOT) else MODEL_ID

if LOCAL_FILES_ONLY and not LOCAL_SNAPSHOT:
    warnings.warn(
        "⚠️ snapshot محلی مدل پیدا نشد. اگر اینترنت/دانلود غیرفعال باشد، لود کردن شکست می‌خورد.\n"
        "راه‌حل: HF_HOME یا HF_HUB_CACHE را به مسیر کش واقعی‌ات ست کن، یا LOAD_REF را دستی مسیر بده."
    )

tokenizer = AutoTokenizer.from_pretrained(LOAD_REF, local_files_only=LOCAL_FILES_ONLY)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    LOAD_REF,
    device_map="auto",
    quantization_config=bnb_config,
    torch_dtype=compute_dtype,
    low_cpu_mem_usage=True,
    local_files_only=LOCAL_FILES_ONLY,
)
model.eval()
model.config.pad_token_id = tokenizer.pad_token_id

MAX_CTX = int(getattr(model.config, "max_position_embeddings", 8192))

def get_input_device(m):
    try:
        return m.get_input_embeddings().weight.device
    except Exception:
        return next(m.parameters()).device

@torch.inference_mode()
def generate_after_marker(prompt: str, marker: str, max_new_tokens: int):
    reserve = max_new_tokens + 16
    max_in = max(64, MAX_CTX - reserve)

    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=max_in)
    dev = get_input_device(model)
    inputs = {k: v.to(dev) for k, v in inputs.items()}

    out = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
        use_cache=True
    )
    txt = tokenizer.decode(out[0], skip_special_tokens=True)
    if marker in txt:
        return txt.split(marker, 1)[1].strip()
    return txt.strip()

# ----------------------------
# 6) Prompt templates
# ----------------------------
PERT_MARKER = "A:"

def prompt_treated_single(cell_line: str, baseline: str, drug: str, K_in: int, K_out: int):
    base_trim = " ".join(baseline.split()[:K_in])
    return (
        "Q: Predict the treated cell sentence after perturbation.\n"
        "Context:\n"
        "- Organism: Homo sapiens\n"
        f"- Cell line: {cell_line}\n"
        f"- Baseline (untreated genes ranked by expression, highest to lowest, top-{K_in}): {base_trim}\n"
        f"- Perturbation: drug = {drug}; time = 24h\n"
        f"Instruction: Return ONLY {K_out} human gene symbols, space-separated, ranked by expression level from HIGHEST to LOWEST. No explanation.\n"
        f"{PERT_MARKER}"
    )

def prompt_treated_combo(cell_line: str, baseline: str, d1: str, d2: str, K_in: int, K_out: int):
    base_trim = " ".join(baseline.split()[:K_in])
    return (
        "Q: Predict the treated cell sentence after combined perturbation.\n"
        "Context:\n"
        "- Organism: Homo sapiens\n"
        f"- Cell line: {cell_line}\n"
        f"- Baseline (untreated genes ranked by expression, highest to lowest, top-{K_in}): {base_trim}\n"
        f"- Perturbation: drugs = [{d1}, {d2}]; time = 24h\n"
        f"Instruction: Return ONLY {K_out} human gene symbols, space-separated, ranked by expression level from HIGHEST to LOWEST. No explanation.\n"
        f"{PERT_MARKER}"
    )


def predict_treated(cell_line: str, baseline: str, drug: str, K_in: int, K_out: int) -> str:
    p = prompt_treated_single(cell_line, baseline, drug, K_in, K_out)
    raw = generate_after_marker(p, PERT_MARKER, max_new_tokens=K_out + 64)
    genes = sanitize_sentence(raw, limit=None)
    return ensure_exact_k_genes(genes, baseline, K_out)

def predict_combo_direct(cell_line: str, baseline: str, d1: str, d2: str, K_in: int, K_out: int) -> str:
    p = prompt_treated_combo(cell_line, baseline, d1, d2, K_in, K_out)
    raw = generate_after_marker(p, PERT_MARKER, max_new_tokens=K_out + 64)
    genes = sanitize_sentence(raw, limit=None)
    return ensure_exact_k_genes(genes, baseline, K_out)

# ----------------------------
# 7) Main loop: rows 0..100 with resume
# ----------------------------
end_eff = min(END_ROW, len(df) - 1)
resume_from = compute_resume_start(START_ROW, end_eff, OUTPUT_CSV, STATE_JSON)

if resume_from > end_eff:
    print(f"🎉 Already finished rows {START_ROW}..{end_eff} (resume_from={resume_from}).")
    raise SystemExit

print(f"✅ Will process rows {resume_from}..{end_eff} (resume-safe)")

file_exists = os.path.exists(OUTPUT_CSV)

# cache برای اینکه اگر baseline تکراری بود، دوباره حساب نکنیم
cache_single = {}   # key: (baseline_hash, drug) -> treated_sentence
cache_combo  = {}   # key: (baseline_hash, d1, d2) -> treated_direct

for ix in tqdm(range(resume_from, end_eff + 1), desc=f"C2S27B rows {START_ROW}-{end_eff}"):
    row = df.iloc[ix]

    cell_name = str(row.get("cell", ""))
    d1_name   = str(row.get("drug1_name", ""))
    d2_name   = str(row.get("drug2_name", ""))

    baseline = sanitize_sentence(row.get("cell_sentence", ""), limit=K_IN)
    sent_hash = hashlib.md5(baseline.encode("utf-8")).hexdigest()

    # drug1
    t1 = ""
    if d1_name:
        key1 = (sent_hash, d1_name)
        if key1 in cache_single:
            t1 = cache_single[key1]
        else:
            try:
                t1 = predict_treated(cell_name, baseline, d1_name, K_in=K_IN, K_out=K_OUT) if baseline else ""
            except Exception as e:
                warnings.warn(f"drug1 failed idx={ix}: {e}")
                t1 = ""
            cache_single[key1] = t1

    # drug2
    t2 = ""
    if d2_name:
        key2 = (sent_hash, d2_name)
        if key2 in cache_single:
            t2 = cache_single[key2]
        else:
            try:
                t2 = predict_treated(cell_name, baseline, d2_name, K_in=K_IN, K_out=K_OUT) if baseline else ""
            except Exception as e:
                warnings.warn(f"drug2 failed idx={ix}: {e}")
                t2 = ""
            cache_single[key2] = t2

    # direct combo
    t12_dir = ""
    if d1_name and d2_name:
        key12 = (sent_hash, d1_name, d2_name)
        if key12 in cache_combo:
            t12_dir = cache_combo[key12]
        else:
            try:
                t12_dir = predict_combo_direct(cell_name, baseline, d1_name, d2_name, K_in=K_IN, K_out=K_OUT) if baseline else ""
            except Exception as e:
                warnings.warn(f"combo direct failed idx={ix}: {e}")
                t12_dir = ""
            cache_combo[key12] = t12_dir

    # خروجی فقط ستون‌هایی که گفتی
    out_row = {
        "orig_index": ix,
        "cell": cell_name,
        "drug1_name": d1_name,
        "drug1_smiles": row.get("drug1_smiles", ""),
        "drug2_name": d2_name,
        "drug2_smiles": row.get("drug2_smiles", ""),
        "label": row.get("label", np.nan),

        "cell_sentence": baseline,
        "c2s_treated_d1_sentence": t1,
        "c2s_treated_d2_sentence": t2,
        "c2s_treated_d12_direct": t12_dir,
    }

    # --- Commit هر سطر: append + flush + fsync + update state.json
    out_df = pd.DataFrame([out_row])
    mode = "a" if file_exists else "w"
    with open(OUTPUT_CSV, mode, encoding="utf-8", newline="") as f:
        out_df.to_csv(f, index=False, header=(not file_exists))
        f.flush()
        os.fsync(f.fileno())

    file_exists = True

    atomic_write_json(STATE_JSON, {
        "last_committed": int(ix),
        "output_csv": OUTPUT_CSV,
        "start_row": int(START_ROW),
        "end_row": int(end_eff),
    })

    # housekeeping
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    gc.collect()

print("✅ Done. Saved:", OUTPUT_CSV)
print("✅ State:", STATE_JSON)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 123.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 85.7 MB/s  0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.4.1
    Uninstalling huggingface_hub-1.4.1:
      Successfully uninstalled huggingface_hub-1.4.1━━━━━━━━━━━━━━ 1/3 [huggingface-hub]
  Attempting uninstall: transformers━━━━━━━━━━━━━━━━━━━━━━━━━━ 1/3 [huggingface-hub]
    Found existing installation: transformers 5.1.0━━━━━━━━━━━ 1/3 [huggingface-hub]
    Uninstalling transformers-5.1.0:0m╸━━━━━━━━━━━━━ 2/3 [transformers]
      Successfully uninstalled transformers-5.1.0━━━━━━━━━━━━━ 2/3 [transformers]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [transformers] [transformers]


MODEL: vandijklab/C2S-Scale-Gemma-2-27B
INPUT: df_with_c2s.csv.gz
OUTPUT: c2s27b_out_0_4396_in300_out300.csv
ROWS: 0 .. 4396 | K_IN: 300 K_OUT: 300
✅ df loaded. shape: (4397, 7) | encoding used: utf-8-sig
columns: ['drug1_name', 'drug2_name', 'label', 'cell_sentence', 'cell', 'drug1_smiles', 'drug2_smiles']


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/4.95M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/555 [00:00<?, ?B/s]

...

model-00012-of-00012.safetensors:   0%|          | 0.00/680M [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/12 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

✅ Will process rows 3670..4396 (resume-safe)


C2S27B rows 0-4396:   0%|          | 0/727 [00:00<?, ?it/s]

In [ ]:
import pandas as pd

path = "c2s27b_out_0_4396_in300_out300.csv"
df_out = pd.read_csv(path)  # ✅ بدون compression

df_out.head(5)


In [ ]:
import pandas as pd

# خواندن فایل CSV معمولی

path = "c2s27b_out_0_4396_in300_out300.csv"
df_out = pd.read_csv(path)

# ذخیره به صورت فشرده gzip
output_path = "c2s27b_out_0_4396_in300_out300.csv.gz"
df_out.to_csv(output_path, compression='gzip', index=False)

print(f"✅ فایل با موفقیت به {output_path} ذخیره شد")
print(f"📊 تعداد ردیف‌ها: {len(df_out)}")


In [ ]:
# محاسبه طول‌ها (مشابه کد قبلی)
df_out["len_cell_sentence"] = df_out["cell_sentence"].fillna("").astype(str).str.split().str.len()

# بررسی ستون‌های خروجی
out_cols = []
for possible_col in ["c2s_treated_d1_sentence", "c2s_treated_d2_sentence", 
                     "c2s_treated_d12_additive", "c2s_treated_d12_direct"]:
    if possible_col in df_out.columns:
        out_cols.append(possible_col)
        df_out[f"len_{possible_col}"] = df_out[possible_col].fillna("").astype(str).str.split().str.len()

print(f"\n✅ ستون‌های خروجی موجود: {out_cols}")

# نمایش طول‌ها
if out_cols:
    show_cols = ["orig_index", "cell", "len_cell_sentence"] + [f"len_{c}" for c in out_cols]
    display(df_out[show_cols].head(10))
    
    print("\n=== آمار طول جملات ===")
    display(df_out[[f"len_{c}" for c in ["cell_sentence"] + out_cols]].describe())


In [ ]:
import pandas as pd

path = "c2s27b_out_0_4396_in300_out300.csv"
df_out = pd.read_csv(path)

df_out.head(5)